The notebook tests blender rendering pipeline

In [ ]:
import json

import torch

%load_ext autoreload
%autoreload 2

# we assume plibs is outside blender_rendering/ and in the same root_dir
# sys.path.insert(0, "../../..")
import os
import shutil
import sys

import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
import trimesh

from blender_rendering import blender_open3d_utils, blender_plib_utils, utils
from plibs import exr_utils, json_utils, render as p_render, rigid_motion, structures, utils as p_utils

repo_root = os.path.abspath("..")
print(f"repo_root: {repo_root}")

In [ ]:
# load mesh

test_mode = "dynamic_two_objs"  # "dynamic"  # static

if test_mode == "static":
    mesh_filename = os.path.join(repo_root, "assets/0a0f1b107acb4b94a8211e11ab69a67f.glb")
    num_frames = 1
    frame_skip = 1
    dynamic = False
elif test_mode == "dynamic":
    mesh_filename = os.path.join(repo_root, "assets/dandys_world_goob_rig__animation.glb")
    num_frames = 4
    frame_skip = 6
    dynamic = True
elif test_mode == "static_two_objs":
    mesh_filename = [
        os.path.join(repo_root, "assets/0a0f1b107acb4b94a8211e11ab69a67f.glb"),
        os.path.join(repo_root, "assets/boom_karts_rocket_vehicle.glb"),
    ]
    num_frames = 1
    frame_skip = 1
    dynamic = False
elif test_mode == "dynamic_two_objs":
    mesh_filename = [
        os.path.join(repo_root, "assets/boom_karts_rocket_vehicle.glb"),
        os.path.join(repo_root, "assets/dandys_world_goob_rig__animation.glb"),
    ]
    num_frames = 4
    frame_skip = 6
    dynamic = True
else:
    raise ValueError(f"Invalid mode: {test_mode}")


#
# t_mesh = trimesh.load_mesh(mesh_filename)
#
# # Compute the bounding box dimensions
# bounds = t_mesh.bounds  # (2 min/max, 3xyz)
# box_length = bounds[1] - bounds[0]  # (3xyz,)
# max_length = box_length.max()  # (,)
# center_xyz = (bounds[1] + bounds[0]) * 0.5  # (3,)
# scaling_factor = 2.0 / max_length
#
# # translate center to origin
# t_mesh.apply_translation(-center_xyz)
#
# # Scale the mesh to [-1, 1]
# t_mesh.apply_scale(scaling_factor)


# generate random cameras
H_c2w_o3d = rigid_motion.generate_random_camera_poses_lookat(
    n=10,
    pinhole_min_r=3,
    pinhole_max_r=5,
    lookat_r=0.1,
    up_method="y",
    invert_y=True,
)  # (n, 4, 4)
H_c2w_o3d = H_c2w_o3d.cpu().numpy()


w, h = 512, 1024
fov = np.random.rand(H_c2w_o3d.shape[0]) * 10 + 30  # (n,)
intrinsic_o3d = torch.from_numpy(
    p_render.derive_camera_intrinsics(
        width_px=w,
        height_px=h,
        fov=fov,
    )
)  # (n, 3, 3)

# add a random cx cy
rcx = w / 2 * (np.random.rand(H_c2w_o3d.shape[0]) - 0.5) * 2 * 0.2  # (n,)
rcy = h / 2 * (np.random.rand(H_c2w_o3d.shape[0]) - 0.5) * 2 * 0.2  # (n,)

intrinsic_o3d[:, 0, 2] += rcx
intrinsic_o3d[:, 1, 2] += rcy

In [ ]:
# construct the config dict
mesh_dicts = []
if isinstance(mesh_filename, str):
    mdict = dict(
        name="mesh",
        filename=mesh_filename,
        normalize_first=True,
        H_c2w=np.eye(4),
        scale=np.array([1.0, 1.0, 1.0]),
    )
    mesh_dicts.append(mdict)
else:
    # randomly place the meshes in the scene
    # blender is z-up and so do most glbs
    for filename in mesh_filename:
        _H_c2w = (
            rigid_motion.generate_random_camera_poses_lookat(
                n=1,
                pinhole_min_r=0,
                pinhole_max_r=1,
                lookat_r=0.25,
                up_method="z",
                invert_y=False,
            )
            .squeeze(0)
            .detach()
            .cpu()
            .numpy()
        )  # (4, 4)
        _scale = np.random.rand(1) * 1.5 + 0.5  # (1,)
        mdict = dict(
            name="mesh",
            filename=filename,
            normalize_first=True,
            H_c2w=_H_c2w,
            scale=[_scale.item() for _ in range(3)],
        )
        mesh_dicts.append(mdict)


camera_dicts = []
for ii in range(H_c2w_o3d.shape[0]):
    mdict = blender_open3d_utils.convert_open3d_camera_to_blender(
        H_c2w=H_c2w_o3d[ii],
        intrinsic=intrinsic_o3d[ii],
        width_px=w,
        height_px=h,
    )
    camera_dicts.append(mdict)


light_dicts = []
mdict = dict(
    name="light 1",
    light_type="SUN",
    H_c2w=[
        [1.0, 0.0, 0.0, 0],
        [0.0, 1.0, 0.0, 0],
        [0.0, 0.0, 1.0, 5.0],
        [0.0, 0.0, 0.0, 1.0],
    ],
    energy=5,
    use_shadow=False,
    specular_factor=1.0,
)
light_dicts.append(mdict)

config_dict = dict(
    meshes=mesh_dicts,
    cameras=camera_dicts,
    lighting=light_dicts,
)

json_filename = "tmp.json"
with open(json_filename, "w") as f:
    json.dump(config_dict, f, indent=2, cls=json_utils.NumpyJsonEncoder)

In [ ]:
# render with blender
out_dir = "tmp_output"

if os.path.exists(out_dir):
    shutil.rmtree(out_dir)


blender_script_version = "v3"
if blender_script_version == "v1":
    blender_script_name = utils.get_blender_utils_path()
elif blender_script_version == "v2":
    blender_script_name = utils.get_blender_utils_v2_path()
elif blender_script_version == "v3":
    blender_script_name = utils.get_blender_utils_v3_path()
else:
    raise NotImplementedError(f"blender_script_version = {blender_script_version} not implemented")


blender_cmd = utils.get_blender_exe()
cmd = (
    f"{blender_cmd} --background --python {blender_script_name} -- "
    f"--filename {json_filename} --out_dir {out_dir} "
    f"--num_frames {num_frames} "
    f"--frame_skip {frame_skip} "
    f"--dynamic {int(dynamic)} "
    f"--normalize_entire_scene 1"
)

print(cmd)
os.system(cmd)
print(f"output is written to {out_dir}")

In [ ]:
# read blender rendered results
# use plibs
rgbd = blender_plib_utils.read_blender_results_to_rgbd(
    result_dir=out_dir,
    from_idx=0,
    to_idx=None,
    from_bidx=0,
    to_bidx=None,
    use_srgb=True,
    flag_save_space=False,
    dynamic=dynamic,
    th_alpha=0.5,
    min_depth=0.0,
    max_depth=1.0e4,
    fps=24,
)
print(f"rgbd.shape: {rgbd.shape}")

# If plibs is not available
#
# all_results = []
# # for ii in range(2): # range(H_c2w_o3d.shape[0]):
# for ii in range(H_c2w_o3d.shape[0]):
#     # exr
#     ddict = dict()
#     for key in ["rgb", "depth", "normal", "obj_id"]:
#         filename = os.path.join(out_dir, f"{ii:04d}_{key}.exr")
#         # print(f"reading {os.path.abspath(filename)}")
#         arr = exr_utils.read_exr(filename)  # (h, w, c)
#         assert arr.shape[:2] == (h, w)
#         ddict[key] = arr
#
#     # srgb
#     filename = os.path.join(out_dir, f"{ii:04d}_srgb.png")
#     arr = iio.imread(filename)
#     if arr.dtype == np.uint8:
#         arr = arr.astype(np.float32) / 255
#     elif arr.dtype == np.uint16:
#         arr = arr.astype(np.float32) / 65535
#     else:
#         raise NotImplementedError
#     assert arr.shape[:2] == (h, w)
#     ddict["srgb"] = arr
#
#     # camera
#     filename = os.path.join(out_dir, f"{ii:04d}_camera.json")
#     with open(filename, "r") as f:
#         cam_info = json.load(f)
#     ddict["cam_info"] = cam_info
#
#     # # # check
#     assert np.allclose(cam_info["H_c2w_open3d"], H_c2w_o3d[ii], rtol=1e-6, atol=1e-6), (
#         f"{cam_info['H_c2w_open3d']} \n\n {H_c2w_o3d[ii]}"
#     )
#     assert np.allclose(cam_info["intrinsic_open3d"], intrinsic_o3d[ii], rtol=1e-6, atol=1e-6), (
#         f"{cam_info['intrinsic_open3d']} \n\n {intrinsic_o3d[ii]}"
#     )
#
#     all_results.append(ddict)
#
# # create rgbd image from all_results
# rgbds = []
# for ii in range(len(all_results)):
#     ddict = all_results[ii]
#     hit_map = torch.from_numpy(ddict["rgb"][:, :, 3]) > 0.5  # (h, w)
#     rgb = torch.from_numpy(ddict["rgb"][:, :, :3]).float()
#     rgb = rgb * hit_map.unsqueeze(-1).float()
#     normal_w = torch.from_numpy(ddict["normal"][:, :, :3]).float()
#     normal_w = normal_w * hit_map.unsqueeze(-1).float()
#     depth = torch.from_numpy(ddict["depth"]).float()
#     depth = depth * hit_map.unsqueeze(-1).float()
#
#     assert (depth > -1e-9).all()
#     cam_info = ddict["cam_info"]
#     camera = structures.Camera(
#         H_c2w=torch.tensor(cam_info["H_c2w_open3d"]).float().reshape(1, 1, 4, 4),
#         intrinsic=torch.tensor(cam_info["intrinsic_open3d"]).float().reshape(1, 1, 3, 3),
#         width_px=cam_info["width_px"],
#         height_px=cam_info["height_px"],
#     )
#
#     rgbd = structures.RGBDImage(
#         rgb=rgb.reshape(1, 1, h, w, 3),
#         depth=depth.reshape(1, 1, h, w),
#         normal_w=normal_w.reshape(1, 1, h, w, 3),
#         hit_map=hit_map.reshape(1, 1, h, w),
#         camera=camera,
#     )
#
#     rgbds.append(rgbd)
#
# rgbd = structures.RGBDImage.cat(rgbds, dim=1)  # (b=1, q, h, w)

In [ ]:
# plot point cloud
world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1)
cam_frames = rgbd.camera.get_camera_frames()[0]
point_cloud = rgbd.get_pcd()

o3d_pcds = point_cloud.get_o3d_pcds(use_normal_for_color=True)

p_utils.visualize_mesh_sequence(meshes=o3d_pcds, static_meshes=[world_frame] + cam_frames)